# Mecânica dos Fluidos

## Aula 02 — De Navier–Stokes aos Métodos Numéricos

### Conservação da massa, Teorema da Divergência, Equação de Poisson, Diferenças Finitas e Elementos Finitos

---

### Objetivos

Nesta aula, vamos construir um caminho que parte das equações fundamentais da Mecânica dos Fluidos e chega à solução numérica de uma equação diferencial parcial.

Ao final, esperamos ser capazes de:

- interpretar os principais termos da equação de Navier–Stokes;
- compreender a origem da aceleração material;
- obter a equação da conservação da massa a partir de um balanço em um volume de controle;
- recuperar e interpretar o Teorema da Divergência;
- reconhecer gradiente, divergente e Laplaciano em diferentes modelos físicos;
- realizar análise dimensional de equações, observando sua compatibilidade;
- perceber que fenômenos físicos distintos podem ser descritos pela mesma estrutura matemática;
- formular um problema de Poisson;
- discretizar a equação de Poisson pelo Método das Diferenças Finitas (MDF);
- montar e resolver o sistema linear correspondente com NumPy;
- introduzir a formulação variacional e a ideia fundamental do Método dos Elementos Finitos (MEF).

---

### Roteiro

$$
\boxed{
\text{Navier--Stokes}
\longrightarrow
\text{Conservação da massa}
\longrightarrow
\text{Teorema da Divergência}
}
$$

$$
\boxed{
\text{Gradiente, divergente e Laplaciano}
\longrightarrow
\text{Poisson}
\longrightarrow
\begin{cases}
\text{MDF}\\
\text{MEF}
\end{cases}
}
$$

> **Ideia central:** começaremos com a física, identificaremos a estrutura matemática do problema e, somente depois, construiremos sua aproximação numérica.

In [1]:
# ============================================================
# Bibliotecas utilizadas ao longo do notebook
# ============================================================

# Computação numérica
import numpy as np

# Computação simbólica
import sympy as sp

# Visualização
import matplotlib.pyplot as plt

# Utilitário para exibição de expressões simbólicas
from IPython.display import display, Markdown

# Configuração do SymPy para exibição matemática
sp.init_printing()

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


# 1. O ponto de partida: as equações de Navier–Stokes

Considere um fluido Newtoniano. A equação da quantidade de movimento pode ser escrita, sob hipóteses usuais, como

$$
\rho
\left[
\frac{\partial \mathbf{u}}{\partial t}
+
(\mathbf{u}\cdot\nabla)\mathbf{u}
\right]
=
-\nabla p
+
\mu\Delta\mathbf{u}
+
\rho\mathbf{f}.
$$

Antes de tentar resolver essa equação, vamos procurar entender **o significado de cada termo**.

Podemos lê-la, de maneira esquemática, como

$$
\boxed{
\text{massa por unidade de volume}
\times
\text{aceleração}
=
\text{forças por unidade de volume}.
}
$$

No lado esquerdo aparece a aceleração do fluido:

$$
\frac{\partial \mathbf{u}}{\partial t}
+
(\mathbf{u}\cdot\nabla)\mathbf{u}.
$$

No lado direito aparecem diferentes mecanismos que podem alterar seu movimento:

$$
\underbrace{-\nabla p}_{\text{pressão}}
+
\underbrace{\mu\Delta\mathbf{u}}_{\text{viscosidade cinemática}}
+
\underbrace{\rho\mathbf{f}}_{\text{forças de corpo}}.
$$

Portanto,

$$
\boxed{
\underbrace{
\rho\left[
\frac{\partial \mathbf{u}}{\partial t}
+
(\mathbf{u}\cdot\nabla)\mathbf{u}
\right]
}_{\text{inércia}}
=
\underbrace{-\nabla p}_{\text{pressão}}
+
\underbrace{\mu\Delta\mathbf{u}}_{\text{viscosidade cinemática}}
+
\underbrace{\rho\mathbf{f}}_{\text{forças de corpo}}.
}
$$

Uma primeira questão importante aparece imediatamente:

> **Por que a aceleração de uma partícula de fluido não é simplesmente**
>
> $$
> \frac{\partial\mathbf{u}}{\partial t}\;?
> $$

Para responder a essa pergunta, precisamos distinguir **o campo de velocidades** da **trajetória de uma partícula de fluido**.

## 1.1 Campo de velocidade e trajetória de uma partícula

Uma distinção fundamental em Mecânica dos Fluidos é a diferença entre:

- observar o **campo de velocidades** em pontos fixos do espaço;
- acompanhar uma **partícula de fluido** enquanto ela se movimenta.

Em duas dimensões, o campo de velocidades é

$$
\mathbf{u}(x,y,t)
=
\begin{pmatrix}
u(x,y,t)\\
v(x,y,t)
\end{pmatrix}.
$$

Isso significa que, para cada posição $(x,y)$ e instante $t$, o campo fornece a velocidade do fluido naquele ponto.

Agora considere uma partícula específica de fluido.

Sua posição varia com o tempo e será representada por

$$
\mathbf{X}(t)
=
\begin{pmatrix}
X(t)\\
Y(t)
\end{pmatrix}.
$$

Observe a diferença de notação:

$$
\boxed{
(x,y)
\quad\text{: coordenadas de um ponto do espaço}
}
$$

enquanto

$$
\boxed{
(X(t),Y(t))
\quad\text{: posição de uma partícula que se move}
}
$$

Por definição, a velocidade da partícula é a derivada de sua posição:

$$
\frac{d\mathbf{X}}{dt}
=
\begin{pmatrix}
\dfrac{dX}{dt}\\[2mm]
\dfrac{dY}{dt}
\end{pmatrix}.
$$

Mas a velocidade dessa partícula deve ser igual ao campo de velocidades avaliado na posição em que ela se encontra:

$$
\frac{d\mathbf{X}}{dt}
=
\mathbf{u}(X(t),Y(t),t).
$$

Portanto,

$$
\boxed{
\frac{dX}{dt}
=
u(X(t),Y(t),t)
}
$$

e

$$
\boxed{
\frac{dY}{dt}
=
v(X(t),Y(t),t).
}
$$

Ou, de forma abreviada,

$$
\boxed{
\dot X=u,
\qquad
\dot Y=v.
}
$$

> O campo $\mathbf{u}(x,y,t)$ descreve a velocidade em todos os pontos do espaço.  
> A trajetória $\mathbf{X}(t)$ descreve o movimento de uma partícula particular através desse campo.

## 1.2 De onde vem a aceleração convectiva?

Queremos agora calcular a aceleração da partícula.

A partícula que estamos acompanhando experimenta a velocidade

$$
\mathbf{u}(X(t),Y(t),t).
$$

Portanto, sua aceleração é

$$
\frac{D\mathbf{u}}{Dt}
=
\frac{d}{dt}
\mathbf{u}(X(t),Y(t),t).
$$

A velocidade depende do tempo de duas maneiras:

1. **explicitamente**, através da variável $t$;
2. **indiretamente**, porque a partícula muda de posição, isto é, $X=X(t)$ e $Y=Y(t)$.

Aplicando a regra da cadeia,

$$
\frac{D\mathbf{u}}{Dt}
=
\frac{\partial\mathbf{u}}{\partial t}
+
\frac{\partial\mathbf{u}}{\partial x}\frac{dX}{dt}
+
\frac{\partial\mathbf{u}}{\partial y}\frac{dY}{dt}.
$$

Mas já sabemos que

$$
\frac{dX}{dt}=u,
\qquad
\frac{dY}{dt}=v.
$$

Logo,

$$
\frac{D\mathbf{u}}{Dt}
=
\frac{\partial\mathbf{u}}{\partial t}
+
u\frac{\partial\mathbf{u}}{\partial x}
+
v\frac{\partial\mathbf{u}}{\partial y}.
$$

Como

$$
\mathbf{u}\cdot\nabla
=
u\frac{\partial}{\partial x}
+
v\frac{\partial}{\partial y},
$$

podemos escrever

$$
\boxed{
\frac{D\mathbf{u}}{Dt}
=
\frac{\partial\mathbf{u}}{\partial t}
+
(\mathbf{u}\cdot\nabla)\mathbf{u}.
}
$$

Temos, portanto, dois mecanismos diferentes de aceleração:

$$
\boxed{
\underbrace{
\frac{\partial\mathbf{u}}{\partial t}
}_{\text{aceleração local}}
+
\underbrace{
(\mathbf{u}\cdot\nabla)\mathbf{u}
}_{\text{aceleração convectiva}}
}
$$

### Interpretação física

A **aceleração local** ocorre quando a velocidade em um ponto fixo do espaço varia com o tempo.

A **aceleração convectiva** ocorre quando uma partícula se desloca para regiões do escoamento onde a velocidade é diferente.

Assim, mesmo que

$$
\frac{\partial\mathbf{u}}{\partial t}=0,
$$

uma partícula ainda pode estar acelerando.

### Exemplo: escoamento estacionário com aceleração

Considere o campo de velocidades bidimensional

$$
\mathbf{u}(x,y)
=
\begin{pmatrix}
2x\\
0
\end{pmatrix}.
$$

Esse campo é **estacionário**, pois não depende explicitamente do tempo. Portanto,

$$
\frac{\partial\mathbf{u}}{\partial t}
=
\begin{pmatrix}
0\\
0
\end{pmatrix}.
$$

Isso, entretanto, não significa que uma partícula de fluido tenha aceleração nula.

O termo convectivo é

$$
(\mathbf{u}\cdot\nabla)\mathbf{u}.
$$

Como

$$
\mathbf{u}
=
\begin{pmatrix}
u\\
v
\end{pmatrix}
=
\begin{pmatrix}
2x\\
0
\end{pmatrix},
$$

temos

$$
\mathbf{u}\cdot\nabla
=
u\frac{\partial}{\partial x}
+
v\frac{\partial}{\partial y}
=
2x\frac{\partial}{\partial x}.
$$

Aplicando esse operador ao campo de velocidades,

$$
(\mathbf{u}\cdot\nabla)\mathbf{u}
=
2x\frac{\partial}{\partial x}
\begin{pmatrix}
2x\\
0
\end{pmatrix}.
$$

Como

$$
\frac{\partial}{\partial x}
\begin{pmatrix}
2x\\
0
\end{pmatrix}
=
\begin{pmatrix}
2\\
0
\end{pmatrix},
$$

segue que

$$
\boxed{
(\mathbf{u}\cdot\nabla)\mathbf{u}
=
\begin{pmatrix}
4x\\
0
\end{pmatrix}.
}
$$

Portanto, a aceleração material é

$$
\boxed{
\frac{D\mathbf{u}}{Dt}
=
\begin{pmatrix}
4x\\
0
\end{pmatrix}.
}
$$

Temos assim um exemplo importante:

$$
\boxed{
\frac{\partial\mathbf{u}}{\partial t}=0
\qquad\text{mas}\qquad
\frac{D\mathbf{u}}{Dt}\neq 0.
}
$$

O campo de velocidades não muda com o tempo, mas a partícula se desloca na direção $x$ e encontra velocidades cada vez maiores.

> **Interpretação:** uma partícula que se move para a direita passa por regiões em que a velocidade $u=2x$ é progressivamente maior. Ela acelera não porque o campo esteja mudando no tempo, mas porque está se deslocando através de um campo espacialmente não uniforme.

In [ ]:
# ============================================================
# Exemplo: aceleração convectiva
# u = [2x, 0]^T
# ============================================================

x, y = sp.symbols('x y', real=True)

# Campo de velocidades
u = 2*x
v = 0

velocidade = sp.Matrix([u, v])

# Aceleração convectiva:
# (u · grad)u = (u*u_x + v*u_y, u*v_x + v*v_y)^T

aceleracao_convectiva = sp.Matrix([
    u*sp.diff(u, x) + v*sp.diff(u, y),
    u*sp.diff(v, x) + v*sp.diff(v, y)
])

display(Markdown(r"### Campo de velocidades"))
display(velocidade)

display(Markdown(r"### Aceleração convectiva"))
display(aceleracao_convectiva)

### Campo de velocidades

⎡2⋅x⎤
⎢   ⎥
⎣ 0 ⎦

### Aceleração convectiva

⎡4⋅x⎤
⎢   ⎥
⎣ 0 ⎦

## 1.3 Forma geral do termo convectivo

No exemplo anterior, calculamos a aceleração convectiva para um campo de velocidades específico.

Agora considere um campo bidimensional genérico

$$
\mathbf{u}(x,y,t)
=
\begin{pmatrix}
u(x,y,t)\\
v(x,y,t)
\end{pmatrix}.
$$

O operador

$$
\mathbf{u}\cdot\nabla
$$

é dado por

$$
\mathbf{u}\cdot\nabla
=
u\frac{\partial}{\partial x}
+
v\frac{\partial}{\partial y}.
$$

Aplicando esse operador ao próprio campo de velocidades,

$$
(\mathbf{u}\cdot\nabla)\mathbf{u}
=
\left(
u\frac{\partial}{\partial x}
+
v\frac{\partial}{\partial y}
\right)
\begin{pmatrix}
u\\
v
\end{pmatrix}.
$$

Portanto,

$$
\boxed{
(\mathbf{u}\cdot\nabla)\mathbf{u}
=
\begin{pmatrix}
u\dfrac{\partial u}{\partial x}
+
v\dfrac{\partial u}{\partial y}
\\[3mm]
u\dfrac{\partial v}{\partial x}
+
v\dfrac{\partial v}{\partial y}
\end{pmatrix}.
}
$$

Observe que aparecem produtos como

$$
u\frac{\partial u}{\partial x},
\qquad
v\frac{\partial u}{\partial y},
\qquad
u\frac{\partial v}{\partial x}.
$$

A própria incógnita aparece multiplicando suas derivadas.

Esse é o termo responsável pela **não linearidade** das equações de Navier–Stokes.

Assim, podemos retornar à equação

$$
\rho
\left[
\frac{\partial\mathbf{u}}{\partial t}
+
(\mathbf{u}\cdot\nabla)\mathbf{u}
\right]
=
-\nabla p
+
\mu\Delta\mathbf{u}
+
\rho\mathbf{f}
$$

e reconhecer uma das principais dificuldades matemáticas do problema:

$$
\boxed{
\text{Navier--Stokes é um sistema de equações diferenciais parciais não lineares.}
}
$$

Além disso, as componentes da velocidade e a pressão estão acopladas.

Mas ainda falta uma equação fundamental.

Para um fluido incompressível, devemos também impor

$$
\boxed{
\nabla\cdot\mathbf{u}=0.
}
$$

De onde vem essa condição?

A resposta está na **conservação da massa**.

# 2. Conservação da massa

Considere uma região **fixa** do espaço $\Omega$, denominada **volume de controle**, limitada pela superfície $\partial\Omega$.

A quantidade total de massa contida em $\Omega$ no instante $t$ é

$$
M(t)=\int_\Omega \rho(\mathbf{x},t)\,d\Omega,
$$

onde $\rho$ é a densidade do fluido.

A conservação da massa afirma que:

> **a variação da massa dentro do volume de controle deve ser igual ao negativo do fluxo líquido de massa que atravessa sua fronteira.**

Vamos construir essa expressão começando por um pequeno elemento da fronteira.

Considere um elemento de área $d\Gamma$ da superfície $\partial\Omega$, com vetor normal unitário exterior $\mathbf{n}$.

Durante um pequeno intervalo de tempo $dt$, o fluido possui velocidade

$$
\mathbf{u}.
$$

Entretanto, somente a componente da velocidade **normal à superfície** transporta fluido através da fronteira.

Essa componente é

$$
\boxed{
\mathbf{u}\cdot\mathbf{n}.
}
$$

Durante o intervalo $dt$, a distância percorrida na direção normal é

$$
(\mathbf{u}\cdot\mathbf{n})\,dt.
$$

Consequentemente, o volume de fluido que atravessa o elemento $d\Gamma$ é

$$
\boxed{
dV
=
(\mathbf{u}\cdot\mathbf{n})\,dt\,d\Gamma.
}
$$

Podemos imaginar esse volume como um pequeno prisma:

$$
\text{volume}
=
\underbrace{d\Gamma}_{\text{área da base}}
\times
\underbrace{(\mathbf{u}\cdot\mathbf{n})dt}_{\text{distância normal}}.
$$

Como

$$
dm=\rho\,dV,
$$

a massa que atravessa esse elemento da fronteira durante $dt$ é

$$
dm
=
\rho(\mathbf{u}\cdot\mathbf{n})\,dt\,d\Gamma.
$$

Dividindo por $dt$, obtemos a **taxa de fluxo de massa** através de $d\Gamma$:

$$
\boxed{
\frac{dm}{dt}
=
\rho\,\mathbf{u}\cdot\mathbf{n}\,d\Gamma.
}
$$

O sinal possui uma interpretação imediata:

- se $\mathbf{u}\cdot\mathbf{n}>0$, o fluido está **saindo** de $\Omega$;
- se $\mathbf{u}\cdot\mathbf{n}<0$, o fluido está **entrando** em $\Omega$;
- se $\mathbf{u}\cdot\mathbf{n}=0$, não há fluxo através daquele ponto da fronteira.

Integrando sobre toda a fronteira,

$$
\boxed{
\int_{\partial\Omega}
\rho\,\mathbf{u}\cdot\mathbf{n}\,d\Gamma
}
$$

representa o **fluxo líquido de massa para fora do volume de controle**.

## 2.1 Balanço de massa no volume de controle

A massa total contida no volume de controle é

$$
M(t)
=
\int_\Omega \rho(\mathbf{x},t)\,d\Omega.
$$

Sua taxa de variação é

$$
\frac{dM}{dt}
=
\frac{d}{dt}
\int_\Omega \rho\,d\Omega.
$$

Como $\Omega$ é um **volume de controle fixo**, seus limites não dependem do tempo. Portanto, podemos escrever

$$
\frac{d}{dt}
\int_\Omega \rho\,d\Omega
=
\int_\Omega
\frac{\partial\rho}{\partial t}\,d\Omega.
$$

Por outro lado, vimos que o fluxo líquido de massa para fora do volume é

$$
\int_{\partial\Omega}
\rho\mathbf{u}\cdot\mathbf{n}\,d\Gamma.
$$

A conservação da massa exige que

$$
\boxed{
\text{taxa de acumulação de massa}
+
\text{taxa de saída de massa}
=
0.
}
$$

Logo,

$$
\boxed{
\frac{d}{dt}
\int_\Omega \rho\,d\Omega
+
\int_{\partial\Omega}
\rho\mathbf{u}\cdot\mathbf{n}\,d\Gamma
=
0.
}
$$

Como o domínio é fixo,

$$
\boxed{
\int_\Omega
\frac{\partial\rho}{\partial t}\,d\Omega
+
\int_{\partial\Omega}
\rho\mathbf{u}\cdot\mathbf{n}\,d\Gamma
=
0.
}
$$

Esta é a **forma integral da conservação da massa**.

> **Observação:** a passagem da derivada para dentro da integral foi possível porque estamos trabalhando com um volume de controle fixo. Para um domínio que se move ou se deforma, precisamos de uma forma mais geral do teorema do transporte.

## 2.2 Do fluxo na fronteira ao divergente no interior

O segundo termo da equação de conservação da massa é

$$
\int_{\partial\Omega}
\rho\mathbf{u}\cdot\mathbf{n}\,d\Gamma.
$$

Esse termo mede o fluxo líquido de massa através da fronteira.

O **Teorema da Divergência** permite transformar essa integral sobre a fronteira em uma integral sobre o volume:

$$
\boxed{
\int_{\partial\Omega}
\mathbf{q}\cdot\mathbf{n}\,d\Gamma
=
\int_\Omega
\nabla\cdot\mathbf{q}\,d\Omega.
}
$$

Para a conservação da massa, o campo de fluxo é

$$
\mathbf{q}=\rho\mathbf{u}.
$$

Portanto,

$$
\int_{\partial\Omega}
\rho\mathbf{u}\cdot\mathbf{n}\,d\Gamma
=
\int_\Omega
\nabla\cdot(\rho\mathbf{u})\,d\Omega.
$$

Substituindo na equação de conservação,

$$
\int_\Omega
\frac{\partial\rho}{\partial t}\,d\Omega
+
\int_\Omega
\nabla\cdot(\rho\mathbf{u})\,d\Omega
=
0.
$$

Agrupando as integrais,

$$
\int_\Omega
\left[
\frac{\partial\rho}{\partial t}
+
\nabla\cdot(\rho\mathbf{u})
\right]
d\Omega
=
0.
$$

Como essa relação deve valer para qualquer volume de controle $\Omega$, concluímos que

$$
\boxed{
\frac{\partial\rho}{\partial t}
+
\nabla\cdot(\rho\mathbf{u})
=
0.
}
$$

Esta é a **equação diferencial da conservação da massa**, também chamada de **equação da continuidade**.

---

### Fluido incompressível com densidade constante

Se $\rho$ é constante,

$$
\frac{\partial\rho}{\partial t}=0
$$

e

$$
\nabla\cdot(\rho\mathbf{u})
=
\rho\nabla\cdot\mathbf{u}.
$$

Assim,

$$
\rho\nabla\cdot\mathbf{u}=0.
$$

Como $\rho\neq0$,

$$
\boxed{
\nabla\cdot\mathbf{u}=0.
}
$$

Agora recuperamos a condição que acompanha as equações de Navier--Stokes para um fluido incompressível:

$$
\boxed{
\begin{aligned}
\rho\left[
\frac{\partial\mathbf{u}}{\partial t}
+
(\mathbf{u}\cdot\nabla)\mathbf{u}
\right]
&=
-\nabla p+\mu\Delta\mathbf{u}+\rho\mathbf{f},
\\[2mm]
\nabla\cdot\mathbf{u}&=0.
\end{aligned}
}
$$

A primeira equação expressa a **conservação da quantidade de movimento**.

A segunda expressa a **conservação da massa**.

# 3. Interpretando o Teorema da Divergência em duas dimensões

Antes de prosseguir, vamos interpretar geometricamente o Teorema da Divergência.

Em duas dimensões, considere um campo vetorial

$$
\mathbf{q}(x,y)
=
\begin{pmatrix}
q_1(x,y)\\
q_2(x,y)
\end{pmatrix}.
$$

Seu divergente é definido por

$$
\boxed{
\nabla\cdot\mathbf{q}
=
\frac{\partial q_1}{\partial x}
+
\frac{\partial q_2}{\partial y}.
}
$$

O Teorema da Divergência estabelece que

$$
\boxed{
\int_\Omega
\nabla\cdot\mathbf{q}\,d\Omega
=
\int_{\partial\Omega}
\mathbf{q}\cdot\mathbf{n}\,d\Gamma.
}
$$

O lado esquerdo mede, de forma acumulada, a presença de **fontes ou sumidouros no interior** de $\Omega$.

O lado direito mede o **fluxo líquido que atravessa a fronteira**.

Assim, o teorema estabelece uma relação fundamental:

$$
\boxed{
\text{produção líquida no interior}
=
\text{fluxo líquido para fora pela fronteira}.
}
$$

Vamos verificar essa ideia no domínio

$$
\Omega=[0,1]\times[0,1].
$$

## 3.1 Exemplo 1 — Campo $\mathbf{q}=(x,y)^T$

Considere

$$
\mathbf{q}(x,y)
=
\begin{pmatrix}
x\\
y
\end{pmatrix}.
$$

O divergente é

$$
\nabla\cdot\mathbf{q}
=
\frac{\partial x}{\partial x}
+
\frac{\partial y}{\partial y}
=
2.
$$

Portanto,

$$
\int_\Omega
\nabla\cdot\mathbf{q}\,d\Omega
=
\int_0^1\int_0^1 2\,dx\,dy
=
2.
$$

Vamos agora calcular o fluxo diretamente pela fronteira do quadrado.

### Lado esquerdo: $x=0$

A normal exterior é

$$
\mathbf{n}=(-1,0)^T.
$$

Como

$$
\mathbf{q}=(0,y)^T,
$$

temos

$$
\mathbf{q}\cdot\mathbf{n}=0.
$$

Logo, a contribuição desse lado é zero.

### Lado direito: $x=1$

A normal exterior é

$$
\mathbf{n}=(1,0)^T.
$$

Como

$$
\mathbf{q}=(1,y)^T,
$$

temos

$$
\mathbf{q}\cdot\mathbf{n}=1.
$$

Portanto,

$$
\int_0^1 1\,dy=1.
$$

### Lado inferior: $y=0$

A normal exterior é

$$
\mathbf{n}=(0,-1)^T.
$$

Como

$$
\mathbf{q}=(x,0)^T,
$$

temos

$$
\mathbf{q}\cdot\mathbf{n}=0.
$$

A contribuição é zero.

### Lado superior: $y=1$

A normal exterior é

$$
\mathbf{n}=(0,1)^T.
$$

Como

$$
\mathbf{q}=(x,1)^T,
$$

temos

$$
\mathbf{q}\cdot\mathbf{n}=1.
$$

Portanto,

$$
\int_0^1 1\,dx=1.
$$

Somando as quatro contribuições,

$$
\int_{\partial\Omega}
\mathbf{q}\cdot\mathbf{n}\,d\Gamma
=
0+1+0+1
=
2.
$$

Assim,

$$
\boxed{
\int_\Omega\nabla\cdot\mathbf{q}\,d\Omega
=
2
=
\int_{\partial\Omega}
\mathbf{q}\cdot\mathbf{n}\,d\Gamma.
}
$$

### Interpretação

Como

$$
\nabla\cdot\mathbf{q}=2>0,
$$

cada ponto do domínio comporta-se localmente como uma **fonte**.

O resultado

$$
\int_\Omega\nabla\cdot\mathbf{q}\,d\Omega=2
$$

significa que a produção líquida acumulada no interior do quadrado é igual a $2$.

Consequentemente, o fluxo líquido que sai pela fronteira também deve ser igual a $2$.


## 3.2 Exemplo 2 — Campo $\mathbf{q}=(-x,y)^T$

Considere agora

$$
\mathbf{q}(x,y)
=
\begin{pmatrix}
-x\\
y
\end{pmatrix}.
$$

Seu divergente é

$$
\nabla\cdot\mathbf{q}
=
\frac{\partial(-x)}{\partial x}
+
\frac{\partial y}{\partial y}
=
-1+1
=
0.
$$

Portanto,

$$
\boxed{
\int_\Omega\nabla\cdot\mathbf{q}\,d\Omega=0.
}
$$

Vamos verificar o que acontece na fronteira.

### Lado $x=0$

Temos

$$
\mathbf{q}=(0,y)^T,
\qquad
\mathbf{n}=(-1,0)^T,
$$

e, portanto,

$$
\mathbf{q}\cdot\mathbf{n}=0.
$$

### Lado $x=1$

Temos

$$
\mathbf{q}=(-1,y)^T,
\qquad
\mathbf{n}=(1,0)^T.
$$

Logo,

$$
\mathbf{q}\cdot\mathbf{n}=-1.
$$

Assim,

$$
\int_0^1(-1)\,dy=-1.
$$

O sinal negativo indica que, nesse lado, há **fluxo entrando no domínio**.

### Lado $y=0$

Temos

$$
\mathbf{q}=(-x,0)^T,
\qquad
\mathbf{n}=(0,-1)^T,
$$

de modo que

$$
\mathbf{q}\cdot\mathbf{n}=0.
$$

### Lado $y=1$

Temos

$$
\mathbf{q}=(-x,1)^T,
\qquad
\mathbf{n}=(0,1)^T.
$$

Logo,

$$
\mathbf{q}\cdot\mathbf{n}=1.
$$

Portanto,

$$
\int_0^1 1\,dx=1.
$$

Somando,

$$
\int_{\partial\Omega}
\mathbf{q}\cdot\mathbf{n}\,d\Gamma
=
0-1+0+1
=
0.
$$

Assim,

$$
\boxed{
\int_\Omega\nabla\cdot\mathbf{q}\,d\Omega
=
0
=
\int_{\partial\Omega}
\mathbf{q}\cdot\mathbf{n}\,d\Gamma.
}
$$

### Interpretação importante

Temos

$$
\nabla\cdot\mathbf{q}=0,
$$

mas isso **não significa que o campo seja nulo ou que não exista fluxo**.

Existe fluxo atravessando a fronteira.

O que ocorre é que

$$
\boxed{
\text{fluxo que entra}
=
\text{fluxo que sai}.
}
$$

Neste exemplo,

$$
-1+1=0.
$$

Portanto,

> **divergente nulo significa ausência de produção líquida local, e não ausência de movimento ou de fluxo.**

Essa interpretação será particularmente importante para compreender a condição de incompressibilidade

$$
\nabla\cdot\mathbf{u}=0.
$$

## 3.3 Exemplo 3 — Campo $\mathbf{q}=(x^2,y^2)^T$

Considere

$$
\mathbf{q}(x,y)
=
\begin{pmatrix}
x^2\\
y^2
\end{pmatrix}.
$$

Agora,

$$
\nabla\cdot\mathbf{q}
=
\frac{\partial x^2}{\partial x}
+
\frac{\partial y^2}{\partial y},
$$

de modo que

$$
\boxed{
\nabla\cdot\mathbf{q}=2x+2y.
}
$$

Diferentemente dos exemplos anteriores, o divergente agora **depende da posição**.

No ponto $(0,0)$,

$$
\nabla\cdot\mathbf{q}=0,
$$

enquanto no ponto $(1,1)$,

$$
\nabla\cdot\mathbf{q}=4.
$$

Portanto, a intensidade da fonte varia ao longo do domínio.

Integrando,

$$
\int_\Omega\nabla\cdot\mathbf{q}\,d\Omega
=
\int_0^1\int_0^1(2x+2y)\,dx\,dy.
$$

Temos

$$
\int_0^1\int_0^1(2x+2y)\,dx\,dy
=
2.
$$

Logo, pelo Teorema da Divergência,

$$
\boxed{
\int_{\partial\Omega}
\mathbf{q}\cdot\mathbf{n}\,d\Gamma
=
2.
}
$$

### Comparando os três exemplos

| Campo $\mathbf{q}$ | $\nabla\cdot\mathbf{q}$ | Interpretação |
|---|---:|---|
| $(x,y)^T$ | $2$ | fonte uniforme |
| $(-x,y)^T$ | $0$ | entrada e saída se equilibram |
| $(x^2,y^2)^T$ | $2x+2y$ | intensidade da fonte varia no espaço |

A ideia fundamental é:

$$
\boxed{
\nabla\cdot\mathbf{q}
\quad\text{mede localmente a tendência de geração ou remoção líquida de fluxo.}
}
$$

In [7]:
# ============================================================
# Divergente de campos vetoriais com SymPy
# ============================================================

x, y = sp.symbols('x y', real=True)

# Campos dos exemplos anteriores
q1 = sp.Matrix([x, y])
q2 = sp.Matrix([-x, y])
q3 = sp.Matrix([x**2, y**2])

def divergente_2d(q):
    return sp.diff(q[0], x) + sp.diff(q[1], y)

campos = [q1, q2, q3]

for i, q in enumerate(campos, start=1):
    div_q = sp.simplify(divergente_2d(q))
    
    display(Markdown(f"### Exemplo {i}"))
    display(Markdown(r"Campo $\mathbf{q}$:"))
    display(q)
    display(Markdown(r"Divergente $\nabla\cdot\mathbf{q}$:"))
    display(div_q)

### Exemplo 1

Campo $\mathbf{q}$:

⎡x⎤
⎢ ⎥
⎣y⎦

Divergente $\nabla\cdot\mathbf{q}$:

### Exemplo 2

Campo $\mathbf{q}$:

... (2 outputs left out)

### Exemplo 3

Campo $\mathbf{q}$:

⎡ 2⎤
⎢x ⎥
⎢  ⎥
⎢ 2⎥
⎣y ⎦

Divergente $\nabla\cdot\mathbf{q}$:

# 4. Três operadores fundamentais

Ao longo desta disciplina, três operadores diferenciais aparecerão repetidamente:

$$
\boxed{
\nabla u,
\qquad
\nabla\cdot\mathbf{q},
\qquad
\Delta u.
}
$$

Embora estejam relacionados, eles atuam sobre objetos diferentes e possuem interpretações distintas.

---

## Gradiente

Se $u=u(x,y)$ é um **campo escalar**, seu gradiente é o campo vetorial

$$
\boxed{
\nabla u
=
\begin{pmatrix}
\dfrac{\partial u}{\partial x}\\[2mm]
\dfrac{\partial u}{\partial y}
\end{pmatrix}.
}
$$

O gradiente aponta na direção de maior crescimento de $u$.

---

## Divergente

Se

$$
\mathbf{q}
=
\begin{pmatrix}
q_1\\
q_2
\end{pmatrix}
$$

é um **campo vetorial**, então

$$
\boxed{
\nabla\cdot\mathbf{q}
=
\frac{\partial q_1}{\partial x}
+
\frac{\partial q_2}{\partial y}.
}
$$

O resultado é um campo escalar.

Como vimos anteriormente, o divergente está relacionado à existência local de fontes e sumidouros.

---

## Laplaciano

Se $u=u(x,y)$ é um campo escalar,

$$
\boxed{
\Delta u
=
\frac{\partial^2u}{\partial x^2}
+
\frac{\partial^2u}{\partial y^2}.
}
$$

Mas observe que

$$
\nabla u
$$

é um campo vetorial. Portanto, podemos calcular seu divergente:

$$
\nabla\cdot(\nabla u)
=
\frac{\partial}{\partial x}
\left(\frac{\partial u}{\partial x}\right)
+
\frac{\partial}{\partial y}
\left(\frac{\partial u}{\partial y}\right).
$$

Logo,

$$
\boxed{
\Delta u
=
\nabla\cdot(\nabla u).
}
$$

Essa identidade será fundamental daqui em diante.

Podemos resumir:

$$
\boxed{
\text{campo escalar}
\xrightarrow{\;\nabla\;}
\text{campo vetorial}
\xrightarrow{\;\nabla\cdot\;}
\text{campo escalar}.
}
$$

Ou seja,

$$
\boxed{
u
\xrightarrow{\;\nabla\;}
\nabla u
\xrightarrow{\;\nabla\cdot\;}
\Delta u.
}
$$

In [8]:
# ============================================================
# Gradiente, divergente e Laplaciano com SymPy
# ============================================================

x, y = sp.symbols('x y', real=True)

# Campo escalar
phi = x**2 + x*y + y**2

# Gradiente de phi
grad_phi = sp.Matrix([
    sp.diff(phi, x),
    sp.diff(phi, y)
])

# Laplaciano de phi
lap_phi = sp.diff(phi, x, 2) + sp.diff(phi, y, 2)

# Divergente do gradiente
div_grad_phi = (
    sp.diff(grad_phi[0], x)
    + sp.diff(grad_phi[1], y)
)

display(Markdown(r"### Campo escalar $\phi(x,y)$"))
display(phi)

display(Markdown(r"### Gradiente $\nabla\phi$"))
display(grad_phi)

display(Markdown(r"### Laplaciano $\Delta\phi$"))
display(sp.simplify(lap_phi))

display(Markdown(r"### Divergente do gradiente $\nabla\cdot(\nabla\phi)$"))
display(sp.simplify(div_grad_phi))

### Campo escalar $\phi(x,y)$

### Gradiente $\nabla\phi$

⎡2⋅x + y⎤
⎢       ⎥
⎣x + 2⋅y⎦

### Laplaciano $\Delta\phi$

### Divergente do gradiente $\nabla\cdot(\nabla\phi)$

# 5. A mesma matemática em diferentes fenômenos físicos

Até agora, encontramos o Laplaciano dentro das equações de Navier--Stokes:

$$
\rho
\left[
\frac{\partial\mathbf{u}}{\partial t}
+
(\mathbf{u}\cdot\nabla)\mathbf{u}
\right]
=
-\nabla p
+
\mu\Delta\mathbf{u}
+
\rho\mathbf{f}.
$$

Vamos agora fazer uma mudança de perspectiva.

Em vez de olhar apenas para a Mecânica dos Fluidos, vamos comparar diferentes fenômenos físicos.

Veremos que:

> **a física pode mudar completamente, enquanto a estrutura matemática permanece essencialmente a mesma.**

O ponto de partida será uma lei de fluxo da forma

$$
\boxed{
\mathbf{q}=-K\nabla u.
}
$$

Aqui:

- $u$ é uma grandeza escalar;
- $\nabla u$ representa sua variação espacial;
- $\mathbf{q}$ representa um fluxo;
- $K$ é um parâmetro físico do problema.

> O sinal negativo indica que o fluxo ocorre, em muitos desses fenômenos, no sentido de valores maiores para valores menores de $u$.

Agora considere um balanço estacionário com uma fonte $s$:

$$
-\nabla\cdot\mathbf{q}= -s
$$

ou, de forma equivalente,

$$
\nabla\cdot\mathbf{q}=s,
$$

dependendo da convenção adotada para o sinal da fonte.

Adotando

$$
-\nabla\cdot(K\nabla u)=s,
$$

obtemos a estrutura geral

$$
\boxed{
-\nabla\cdot(K\nabla u)=s.
}
$$

Se $K$ for constante,

$$
-\nabla\cdot(K\nabla u)
=
-K\nabla\cdot(\nabla u).
$$

Como

$$
\nabla\cdot(\nabla u)=\Delta u,
$$

segue que

$$
-K\Delta u=s.
$$

Dividindo por $K$,

$$
\boxed{
-\Delta u=f,
}
$$

onde

$$
f=\frac{s}{K}.
$$

Essa é a **equação de Poisson**.

Vamos agora descobrir quantos problemas físicos diferentes possuem exatamente essa estrutura.

## 5.1 Condução de calor

Considere um campo de temperatura

$$
T=T(x,y).
$$

A **Lei de Fourier** estabelece que o fluxo de calor é proporcional ao negativo do gradiente da temperatura:

$$
\boxed{
\mathbf{q}_T=-k\nabla T,
}
$$

onde $k$ é a condutividade térmica.

O sinal negativo possui uma interpretação física clara:

$$
\boxed{
\text{o calor flui das regiões mais quentes para as mais frias.}
}
$$

Em regime estacionário, considerando uma fonte volumétrica de calor $Q$, o balanço conduz a

$$
-\nabla\cdot(k\nabla T)=Q.
$$

Se o material for homogêneo e isotrópico, com $k$ constante,

$$
-k\Delta T=Q.
$$

Portanto,

$$
\boxed{
-\Delta T=\frac{Q}{k}.
}
$$

Temos uma equação de Poisson.

## 5.2 Difusão de uma espécie

Considere agora a concentração de uma substância

$$
c=c(x,y).
$$

A **Lei de Fick** estabelece

$$
\boxed{
\mathbf{J}=-D\nabla c,
}
$$

onde:

- $\mathbf{J}$ é o fluxo da espécie;
- $D$ é o coeficiente de difusão.

Novamente, o fluxo ocorre das regiões de maior concentração para as de menor concentração.

Em regime estacionário, na presença de uma fonte $S$,

$$
-\nabla\cdot(D\nabla c)=S.
$$

Se $D$ for constante,

$$
-D\Delta c=S.
$$

Portanto,

$$
\boxed{
-\Delta c=\frac{S}{D}.
}
$$

Novamente encontramos uma equação de Poisson.

## 5.3 Eletrostática

Considere o potencial elétrico

$$
\phi=\phi(x,y).
$$

O campo elétrico é relacionado ao potencial por

$$
\boxed{
\mathbf{E}=-\nabla\phi.
}
$$

A Lei de Gauss, na forma diferencial, estabelece

$$
\nabla\cdot(\varepsilon\mathbf{E})=\rho_e,
$$

onde:

- $\varepsilon$ é a permissividade elétrica;
- $\rho_e$ é a densidade de carga elétrica.

Substituindo

$$
\mathbf{E}=-\nabla\phi,
$$

obtemos

$$
-\nabla\cdot(\varepsilon\nabla\phi)=\rho_e.
$$

Se $\varepsilon$ for constante,

$$
-\varepsilon\Delta\phi=\rho_e.
$$

Logo,

$$
\boxed{
-\Delta\phi=\frac{\rho_e}{\varepsilon}.
}
$$

Mais uma vez encontramos a equação de Poisson.

## 5.4 Membrana sob tensão

Considere uma membrana fina submetida a uma tensão uniforme $\tau$.

Se

$$
w=w(x,y)
$$

representa seu deslocamento transversal e $p(x,y)$ uma carga distribuída, o modelo linear de equilíbrio da membrana é

$$
\boxed{
-\tau\Delta w=p.
}
$$

Se $\tau$ for constante,

$$
\boxed{
-\Delta w=\frac{p}{\tau}.
}
$$

Novamente temos uma equação de Poisson.

---

## O que esses problemas têm em comum?

Temos quatro fenômenos físicos distintos:

| Fenômeno | Incógnita | Parâmetro | Fonte |
|---|---|---|---|
| condução de calor | temperatura $T$ | condutividade $k$ | calor $Q$ |
| difusão | concentração $c$ | difusividade $D$ | fonte $S$ |
| eletrostática | potencial $\phi$ | permissividade $\varepsilon$ | carga $\rho_e$ |
| membrana | deslocamento $w$ | tensão $\tau$ | carga $p$ |

Entretanto, matematicamente, todos podem conduzir à estrutura

$$
\boxed{
-\nabla\cdot(K\nabla u)=s.
}
$$

E, quando $K$ é constante,

$$
\boxed{
-\Delta u=f.
}
$$

Portanto,

$$
\boxed{
\text{a física muda; a estrutura matemática permanece.}
}
$$

Essa observação é extremamente importante em métodos numéricos.

Uma vez que aprendemos a resolver numericamente

$$
-\Delta u=f,
$$

o mesmo método pode ser reutilizado para estudar temperatura, concentração, potencial elétrico, deslocamento de uma membrana e muitos outros fenômenos.

O computador não precisa saber se $u$ representa temperatura, concentração, potencial ou deslocamento.

Ele recebe uma **estrutura matemática**.

# 6. Um problema de Poisson com solução conhecida

Vamos considerar o domínio quadrado

$$
\Omega=(0,1)\times(0,1)
$$

e o problema de Poisson

$$
\boxed{
-\Delta u=f
\qquad\text{em }\Omega,
}
$$

com condição de contorno de Dirichlet homogênea

$$
\boxed{
u=0
\qquad\text{em }\partial\Omega.
}
$$

Nosso objetivo será resolver esse mesmo problema por diferentes métodos numéricos e comparar as aproximações obtidas.

Para isso, é conveniente conhecer previamente a solução exata.

Vamos escolher

$$
\boxed{
u(x,y)=x(1-x)y(1-y).
}
$$

Essa função é particularmente conveniente porque satisfaz automaticamente a condição de contorno.

De fato, se $x=0$ ou $x=1$,

$$
u(x,y)=0,
$$

e, se $y=0$ ou $y=1$,

$$
u(x,y)=0.
$$

Portanto,

$$
u=0
\qquad\text{em }\partial\Omega.
$$

Agora fazemos o caminho inverso:

> escolhemos uma solução $u$ e determinamos qual deve ser a fonte $f$ para que essa função seja solução do problema.

Essa estratégia é conhecida como **método da solução manufaturada**.

Ela é muito útil na verificação de métodos numéricos, pois permite comparar diretamente a solução aproximada com uma solução exata conhecida.

## 6.1 Determinação da fonte $f$

Temos

$$
u(x,y)=x(1-x)y(1-y).
$$

Como

$$
-\Delta u=f,
$$

precisamos calcular

$$
f
=
-\left(
\frac{\partial^2u}{\partial x^2}
+
\frac{\partial^2u}{\partial y^2}
\right).
$$

Comecemos pelas derivadas em relação a $x$.

Como

$$
u=x(1-x)y(1-y),
$$

temos

$$
\frac{\partial u}{\partial x}
=
(1-2x)y(1-y).
$$

Derivando novamente,

$$
\frac{\partial^2u}{\partial x^2}
=
-2y(1-y).
$$

Analogamente,

$$
\frac{\partial u}{\partial y}
=
x(1-x)(1-2y)
$$

e

$$
\frac{\partial^2u}{\partial y^2}
=
-2x(1-x).
$$

Portanto,

$$
\Delta u
=
-2y(1-y)-2x(1-x).
$$

Assim,

$$
\boxed{
f(x,y)
=
2\left[
x(1-x)+y(1-y)
\right].
}
$$

Nosso problema completo é, portanto,

$$
\boxed{
\begin{aligned}
-\Delta u
&=
2\left[x(1-x)+y(1-y)\right]
&&\text{em }\Omega,
\\
u&=0
&&\text{em }\partial\Omega.
\end{aligned}
}
$$

e conhecemos sua solução exata:

$$
\boxed{
u_{\mathrm{ex}}(x,y)=x(1-x)y(1-y).
}
$$

In [10]:
# ============================================================
# Solução manufaturada para o problema de Poisson
# ============================================================

x, y = sp.symbols('x y', real=True)

# Solução exata escolhida
u_exata_sym = x*(1 - x)*y*(1 - y)

# Derivadas de primeira ordem
du_dx = sp.diff(u_exata_sym, x)
du_dy = sp.diff(u_exata_sym, y)

# Derivadas de segunda ordem
d2u_dx2 = sp.diff(u_exata_sym, x, 2)
d2u_dy2 = sp.diff(u_exata_sym, y, 2)

# Laplaciano
laplaciano_u = sp.simplify(d2u_dx2 + d2u_dy2)

# Como -Delta u = f
f_sym = sp.simplify(-laplaciano_u)

display(Markdown(r"### Solução exata $u_{\text{ex}}(x,y)$"))
display(u_exata_sym)

display(Markdown(r"### Gradiente $\nabla u$"))
display(sp.Matrix([du_dx, du_dy]))

display(Markdown(r"### Laplaciano $\Delta u$"))
display(laplaciano_u)

display(Markdown(r"### Fonte $f=-\Delta u$"))
display(f_sym)

### Solução exata $u_{\text{ex}}(x,y)$

### Gradiente $\nabla u$

⎡-x⋅y⋅(1 - y) + y⋅(1 - x)⋅(1 - y)⎤
⎢                                ⎥
⎣-x⋅y⋅(1 - x) + x⋅(1 - x)⋅(1 - y)⎦

### Laplaciano $\Delta u$

### Fonte $f=-\Delta u$

## 6.2 Um valor de referência

O centro do domínio é

$$
(x,y)=\left(\frac12,\frac12\right).
$$

A solução exata nesse ponto vale

$$
u\left(\frac12,\frac12\right)
=
\frac12
\left(1-\frac12\right)
\frac12
\left(1-\frac12\right).
$$

Portanto,

$$
\boxed{
u\left(\frac12,\frac12\right)
=
\frac{1}{16}
=
0.0625.
}
$$

A fonte no centro é

$$
f\left(\frac12,\frac12\right)
=
2\left[
\frac12\left(1-\frac12\right)
+
\frac12\left(1-\frac12\right)
\right],
$$

isto é,

$$
\boxed{
f\left(\frac12,\frac12\right)=1.
}
$$

Esses dois valores,

$$
\boxed{
u_C=\frac1{16},
\qquad
f_C=1,
}
$$

serão úteis quando construirmos nossa primeira discretização.

# 7. Método das Diferenças Finitas

Queremos aproximar a equação

$$
-\Delta u=f.
$$

Em duas dimensões,

$$
-\left(
\frac{\partial^2u}{\partial x^2}
+
\frac{\partial^2u}{\partial y^2}
\right)
=
f.
$$

A ideia do Método das Diferenças Finitas é substituir as derivadas por aproximações construídas a partir dos valores da função em pontos de uma malha.

Considere inicialmente uma função de uma variável.

Expandindo $u(x+h)$ em série de Taylor em torno de $x$,

$$
u(x+h)
=
u(x)
+
h u'(x)
+
\frac{h^2}{2}u''(x)
+
\frac{h^3}{6}u'''(x)
+
O(h^4).
$$

Analogamente,

$$
u(x-h)
=
u(x)
-
h u'(x)
+
\frac{h^2}{2}u''(x)
-
\frac{h^3}{6}u'''(x)
+
O(h^4).
$$

Somando as duas expressões,

$$
u(x+h)+u(x-h)
=
2u(x)+h^2u''(x)+O(h^4).
$$

Portanto,

$$
\boxed{
u''(x)
\approx
\frac{
u(x+h)-2u(x)+u(x-h)
}{h^2}.
}
$$

Essa é a aproximação por **diferenças centradas** para a derivada segunda.

O erro dessa aproximação é de ordem

$$
O(h^2).
$$

## 7.1 O Laplaciano em uma malha bidimensional

Considere um ponto da malha

$$
(x_i,y_j).
$$

Denotaremos

$$
U_{i,j}\approx u(x_i,y_j).
$$

Na direção $x$,

$$
\frac{\partial^2u}{\partial x^2}(x_i,y_j)
\approx
\frac{
U_{i+1,j}
-2U_{i,j}
+U_{i-1,j}
}{h^2}.
$$

Na direção $y$,

$$
\frac{\partial^2u}{\partial y^2}(x_i,y_j)
\approx
\frac{
U_{i,j+1}
-2U_{i,j}
+U_{i,j-1}
}{h^2}.
$$

Somando,

$$
\Delta u(x_i,y_j)
\approx
\frac{
U_{i+1,j}
+
U_{i-1,j}
+
U_{i,j+1}
+
U_{i,j-1}
-
4U_{i,j}
}{h^2}.
$$

Como nosso problema é

$$
-\Delta u=f,
$$

obtemos

$$
\boxed{
\frac{
4U_{i,j}
-
U_{i+1,j}
-
U_{i-1,j}
-
U_{i,j+1}
-
U_{i,j-1}
}{h^2}
=
f_{i,j}.
}
$$

Essa expressão é conhecida como **estêncil de cinco pontos**.

Podemos representá-la esquematicamente por

$$
\frac{1}{h^2}
\begin{array}{ccc}
 & -1 & \\
-1 & 4 & -1\\
 & -1 &
\end{array}
$$

Cada equação associa o valor da solução em um ponto aos seus quatro vizinhos:

- leste;
- oeste;
- norte;
- sul.

## 7.2 Nossa primeira malha

Vamos começar com a malha mais simples que possui um ponto interior.

Dividimos cada lado do quadrado em dois intervalos:

$$
h=\frac12.
$$

Os pontos em cada direção são

$$
0,\qquad \frac12,\qquad 1.
$$

Existe apenas **um ponto interior**:

$$
C=
\left(
\frac12,\frac12
\right).
$$

Seus quatro vizinhos pertencem à fronteira.

Como a condição de Dirichlet é homogênea,

$$
u=0
\qquad\text{em }\partial\Omega,
$$

temos

$$
U_E=U_W=U_N=U_S=0.
$$

O estêncil no ponto central fornece

$$
\frac{
4U_C-U_E-U_W-U_N-U_S
}{h^2}
=
f_C.
$$

Portanto,

$$
\frac{4U_C}{(1/2)^2}=1.
$$

Como

$$
\left(\frac12\right)^2=\frac14,
$$

segue que

$$
16U_C=1.
$$

Logo,

$$
\boxed{
U_C^{\mathrm{MDF}}=\frac1{16}=0.0625.
}
$$

Mas já sabemos que

$$
u_{\mathrm{ex}}
\left(
\frac12,\frac12
\right)
=
\frac1{16}.
$$

Assim, nessa malha,

$$
\boxed{
U_C^{\mathrm{MDF}}
=
u_{\mathrm{ex}}
\left(
\frac12,\frac12
\right).
}
$$

### Atenção

Isso **não significa que o MDF seja exato para esse problema**.

Temos uma coincidência particular entre:

- a solução manufaturada escolhida;
- o ponto considerado;
- a malha utilizada;
- o estêncil de cinco pontos.

Para avaliar o método numericamente, precisamos utilizar malhas mais refinadas e estudar o comportamento do erro.

## 7.3 Do estêncil ao sistema linear

Para a malha anterior, havia apenas um ponto interior e obtivemos uma única equação:

$$
16U_C=1.
$$

Podemos escrever essa equação na forma matricial:

$$
\boxed{
A\mathbf{U}=\mathbf{b},
}
$$

com

$$
A=
\begin{pmatrix}
16
\end{pmatrix},
\qquad
\mathbf{U}
=
\begin{pmatrix}
U_C
\end{pmatrix},
\qquad
\mathbf{b}
=
\begin{pmatrix}
1
\end{pmatrix}.
$$

Quando refinamos a malha, passamos a ter vários pontos interiores.

Para cada ponto interior $(i,j)$, escrevemos

$$
\frac{
4U_{i,j}
-U_{i-1,j}
-U_{i+1,j}
-U_{i,j-1}
-U_{i,j+1}
}{h^2}
=
f_{i,j}.
$$

Cada ponto interior fornece **uma equação**.

Cada valor desconhecido $U_{i,j}$ corresponde a **um grau de liberdade**.

Depois de numerarmos os pontos interiores, todas essas equações podem ser reunidas em um único sistema:

$$
\boxed{
A\mathbf{U}=\mathbf{b}.
}
$$

A matriz $A$ contém os coeficientes provenientes do estêncil:

$$
\frac{1}{h^2}
\begin{array}{ccc}
 & -1 & \\
-1 & 4 & -1\\
 & -1 &
\end{array}
$$

Portanto:

- o ponto central contribui com $4/h^2$;
- cada vizinho interior contribui com $-1/h^2$;
- valores conhecidos na fronteira são incorporados ao lado direito.

Para a condição de Dirichlet homogênea utilizada aqui,

$$
u=0\quad\text{em }\partial\Omega,
$$

as contribuições dos pontos da fronteira são nulas.

## 7.4 Numeração dos graus de liberdade

Suponha que cada lado do domínio seja dividido em $N$ intervalos.

Então,

$$
h=\frac{1}{N}.
$$

Em cada direção existem

$$
N+1
$$

pontos da malha.

Como os pontos da fronteira possuem valores conhecidos, as incógnitas estão apenas nos pontos interiores.

Existem

$$
N-1
$$

pontos interiores em cada direção e, portanto,

$$
\boxed{
N_{\mathrm{dof}}=(N-1)^2
}
$$

graus de liberdade.

Para montar uma matriz bidimensional como um sistema linear, precisamos transformar os índices $(i,j)$ em um único índice $k$.

Usaremos

$$
\boxed{
k=(j-1)(N-1)+(i-1),
}
$$

com

$$
i,j=1,\ldots,N-1.
$$

Assim, cada ponto interior $(i,j)$ corresponde a uma posição do vetor de incógnitas $\mathbf{U}$.

In [11]:
# ============================================================
# Solução exata e termo fonte
# ============================================================

def u_exata(x, y):
    """
    Solução exata do problema de Poisson.
    """
    return x*(1 - x)*y*(1 - y)


def fonte(x, y):
    """
    Termo fonte f(x,y) = -Delta u.
    """
    return 2*(x*(1 - x) + y*(1 - y))

In [12]:
# ============================================================
# Método das Diferenças Finitas para -Delta u = f
# u = 0 na fronteira
# ============================================================

def resolve_poisson_mdf(N):
    """
    Resolve

        -Delta u = f    em (0,1)x(0,1)
               u = 0    na fronteira

    usando diferenças finitas centradas.

    Parâmetro
    ---------
    N : int
        Número de intervalos em cada direção.

    Retorna
    -------
    X, Y : coordenadas da malha
    U    : solução numérica incluindo a fronteira
    A    : matriz do sistema linear
    b    : vetor do lado direito
    """

    # Passo da malha
    h = 1.0 / N

    # Número de pontos interiores em cada direção
    n = N - 1

    # Número total de graus de liberdade
    ndof = n*n

    # Matriz e vetor do sistema linear
    A = np.zeros((ndof, ndof))
    b = np.zeros(ndof)

    # --------------------------------------------------------
    # Converte o índice bidimensional (i,j)
    # em um índice k do vetor de incógnitas
    # --------------------------------------------------------
    def idx(i, j):
        return (j - 1)*n + (i - 1)

    # --------------------------------------------------------
    # Montagem do sistema linear
    # --------------------------------------------------------
    for j in range(1, N):
        for i in range(1, N):

            k = idx(i, j)

            # Coordenadas do ponto
            xi = i*h
            yj = j*h

            # Coeficiente central
            A[k, k] = 4.0 / h**2

            # Lado direito
            b[k] = fonte(xi, yj)

            # Vizinho à esquerda
            if i - 1 >= 1:
                A[k, idx(i - 1, j)] = -1.0 / h**2

            # Vizinho à direita
            if i + 1 <= N - 1:
                A[k, idx(i + 1, j)] = -1.0 / h**2

            # Vizinho abaixo
            if j - 1 >= 1:
                A[k, idx(i, j - 1)] = -1.0 / h**2

            # Vizinho acima
            if j + 1 <= N - 1:
                A[k, idx(i, j + 1)] = -1.0 / h**2

    # --------------------------------------------------------
    # Resolução de A U = b
    # --------------------------------------------------------
    U_interior = np.linalg.solve(A, b)

    # --------------------------------------------------------
    # Reconstrução da solução sobre toda a malha
    # A fronteira permanece igual a zero
    # --------------------------------------------------------
    U = np.zeros((N + 1, N + 1))

    for j in range(1, N):
        for i in range(1, N):
            U[j, i] = U_interior[idx(i, j)]

    # Coordenadas da malha
    grid = np.linspace(0.0, 1.0, N + 1)
    X, Y = np.meshgrid(grid, grid)

    return X, Y, U, A, b

In [13]:
# ============================================================
# Primeira malha: N = 2
# ============================================================

N = 2

X, Y, U, A, b = resolve_poisson_mdf(N)

print("Matriz A:")
print(A)

print("\nVetor b:")
print(b)

print("\nSolução numérica U:")
print(U)

print("\nValor no centro:")
print(U[1, 1])

print("\nValor exato no centro:")
print(u_exata(0.5, 0.5))

Matriz A:
[[16.]]

Vetor b:
[1.]

Solução numérica U:
[[0.     0.     0.    ]
 [0.     0.0625 0.    ]
 [0.     0.     0.    ]]

Valor no centro:
0.0625

Valor exato no centro:
0.0625


## 7.5 Comparando o algoritmo com a conta manual

Para $N=2$,

$$
h=\frac12
$$

e existe apenas um grau de liberdade.

Portanto, o programa deve construir

$$
A=
\begin{pmatrix}
16
\end{pmatrix}
$$

e

$$
\mathbf{b}
=
\begin{pmatrix}
1
\end{pmatrix}.
$$

Assim,

$$
A\mathbf{U}=\mathbf{b}
$$

significa simplesmente

$$
16U_C=1.
$$

A solução calculada pelo NumPy é

$$
\boxed{
U_C=0.0625=\frac1{16}.
}
$$

Ou seja, o computador realizou exatamente a mesma operação que fizemos anteriormente à mão.

Essa observação é importante:

> **O método numérico não está escondido no comando `np.linalg.solve(A, b)`.**

O comando `np.linalg.solve(A, b)` apenas resolve o sistema linear

$$
A\mathbf{U}=\mathbf{b}.
$$

O **Método das Diferenças Finitas** apareceu antes, quando substituímos as derivadas por diferenças finitas e, a partir do estêncil resultante, construímos a matriz $A$ e o vetor $\mathbf{b}$.

Portanto, podemos separar duas etapas:

$$
\boxed{
\text{discretização da EDP}
\longrightarrow
\text{construção de } A\mathbf{U}=\mathbf{b}
}
$$

e depois

$$
\boxed{
\text{álgebra linear}
\longrightarrow
\text{solução de } A\mathbf{U}=\mathbf{b}.
}
$$

Em outras palavras,

$$
\boxed{
\text{EDP}
\longrightarrow
\text{discretização}
\longrightarrow
A\mathbf{U}=\mathbf{b}
\longrightarrow
\text{solução do sistema}.
}
$$

Essa distinção será importante quando estudarmos o Método dos Elementos Finitos.

No MEF também chegaremos a um sistema linear,

$$
K\mathbf{U}=\mathbf{F},
$$

e poderemos utilizar novamente um resolvedor de sistemas lineares.

O que muda fundamentalmente entre MDF e MEF não é a operação final de resolver o sistema, mas **a maneira como esse sistema é construído**.

# 8. Refinamento da malha e estudo do erro

Até agora utilizamos a malha mais simples possível:

$$
N=2,
\qquad
h=\frac12.
$$

Para essa malha, obtivemos no único ponto interior

$$
U_C=\frac1{16},
$$

que coincide com o valor da solução exata naquele ponto.

Mas um método numérico deve ser analisado observando o que acontece quando a malha é refinada.

Vamos considerar sucessivamente

$$
N=2,\;4,\;8,\;16,\;32.
$$

Como

$$
h=\frac1N,
$$

isso corresponde a uma sequência de malhas cada vez mais finas.

Para cada malha, vamos comparar a solução numérica

$$
U_{i,j}
$$

com a solução exata

$$
u_{\mathrm{ex}}(x_i,y_j).
$$

Podemos medir o erro máximo nos pontos da malha por

$$
\boxed{
E_\infty
=
\max_{i,j}
\left|
U_{i,j}-u_{\mathrm{ex}}(x_i,y_j)
\right|.
}
$$

### Experimento numérico

Para cada valor de $N$:

1. resolva o problema pelo MDF;
2. calcule a solução exata nos pontos da malha;
3. calcule $E_\infty$;
4. registre $h$, o número de graus de liberdade e o erro;
5. observe o comportamento do erro quando $h$ diminui.

Lembre que

$$
N_{\mathrm{dof}}=(N-1)^2.
$$

A pergunta que queremos investigar é:

> **A solução numérica se aproxima da solução exata quando refinamos a malha?**

## 8.1 Refinando a malha

Vamos agora executar automaticamente o mesmo problema para diferentes valores de $N$:

$$
N=2,\;4,\;8,\;16,\;32.
$$

Para cada valor de $N$:

1. construímos a malha;
2. resolvemos o problema pelo MDF;
3. calculamos a solução exata nos mesmos pontos;
4. calculamos o erro em cada ponto da malha;
5. tomamos o maior desses erros.

O erro máximo será

$$
E_\infty
=
\max_{i,j}
\left|
U_{i,j}-u_{\mathrm{ex}}(x_i,y_j)
\right|.
$$

Por exemplo, depois de resolver o problema, teremos duas matrizes:

$$
U=
\begin{pmatrix}
U_{0,0} & \cdots & U_{N,0}\\
\vdots & & \vdots\\
U_{0,N} & \cdots & U_{N,N}
\end{pmatrix}
$$

contendo a solução numérica e

$$
U_{\mathrm{ex}}
=
\begin{pmatrix}
u(x_0,y_0) & \cdots & u(x_N,y_0)\\
\vdots & & \vdots\\
u(x_0,y_N) & \cdots & u(x_N,y_N)
\end{pmatrix}
$$

contendo a solução exata avaliada nos mesmos pontos.

A diferença

$$
|U-U_{\mathrm{ex}}|
$$

fornece o erro em cada nó.

Finalmente,

$$
\boxed{
E_\infty=\max|U-U_{\mathrm{ex}}|
}
$$

seleciona o maior erro nodal da malha.

In [15]:
# ============================================================
# Refinamento da malha e cálculo do erro
# ============================================================

Ns = [2, 4, 8, 16, 32]

resultados = []

for N in Ns:

    # Resolve o problema numericamente
    X, Y, U, A, b = resolve_poisson_mdf(N)

    # Avalia a solução exata nos mesmos pontos da malha
    Ue = u_exata(X, Y)

    # Erro em cada ponto da malha
    erro_nodal = np.abs(U - Ue)

    # Maior erro da malha
    erro_max = np.max(erro_nodal)

    # Tamanho dos elementos da malha
    h = 1.0 / N

    # Número de graus de liberdade
    ndof = (N - 1)**2

    # Guarda os resultados
    resultados.append((N, h, ndof, erro_max))


# Mostra os resultados
print(" N       h       graus de liberdade       erro máximo")
print("-"*60)

for N, h, ndof, erro in resultados:
    print(f"{N:2d}   {h:8.5f}          {ndof:4d}             {erro:.6e}")

 N       h       graus de liberdade       erro máximo
------------------------------------------------------------
 2    0.50000             1             0.000000e+00
 4    0.25000             9             1.387779e-17
 8    0.12500            49             2.081668e-17
16    0.06250           225             4.857226e-17
32    0.03125           961             9.714451e-17


## 8.2 Exercício — Um resultado inesperado

Ao refinar a malha, esperávamos observar a aproximação da solução numérica à solução exata.

Entretanto, os resultados obtidos mostram algo curioso:

| $N$ | $h$ | graus de liberdade | erro máximo |
|---:|---:|---:|---:|
| 2  | 0.50000 | 1   | $0$ |
| 4  | 0.25000 | 9   | $\approx 1.4\times10^{-17}$ |
| 8  | 0.12500 | 49  | $\approx 2.1\times10^{-17}$ |
| 16 | 0.06250 | 225 | $\approx 4.9\times10^{-17}$ |
| 32 | 0.03125 | 961 | $\approx 9.7\times10^{-17}$ |

Os valores da ordem de $10^{-17}$ podem ser interpretados como zero dentro da precisão numérica do computador.

Portanto, aparentemente,

$$
U_{i,j}\approx u(x_i,y_j)
$$

em todos os nós das malhas consideradas.

### Investigue

A solução exata utilizada foi

$$
u(x,y)=x(1-x)y(1-y).
$$

A aproximação por diferenças centradas da derivada segunda em relação a $x$ é

$$
D_{xx}u(x,y)
=
\frac{
u(x+h,y)-2u(x,y)+u(x-h,y)
}{h^2}.
$$

Analogamente,

$$
D_{yy}u(x,y)
=
\frac{
u(x,y+h)-2u(x,y)+u(x,y-h)
}{h^2}.
$$

1. Calcule analiticamente

$$
\frac{\partial^2u}{\partial x^2}
\qquad\text{e}\qquad
\frac{\partial^2u}{\partial y^2}.
$$

2. Calcule $D_{xx}u$ e $D_{yy}u$ usando a fórmula de diferenças finitas.

3. Compare:

$$
D_{xx}u
\quad\text{com}\quad
\frac{\partial^2u}{\partial x^2},
$$

e

$$
D_{yy}u
\quad\text{com}\quad
\frac{\partial^2u}{\partial y^2}.
$$

4. O que você observa?

5. Explique por que o erro nodal obtido numericamente é praticamente zero para todas as malhas testadas.

6. Esse resultado permite concluir que o MDF calcula exatamente a solução de qualquer problema de Poisson? Justifique.

### Desafio

Considere agora outra solução manufaturada:

$$
u(x,y)=\sin(\pi x)\sin(\pi y).
$$

**a)** Determine a função $f(x,y)$ para que

$$
-\Delta u=f.
$$

**b)** Modifique as funções `u_ex` e `fonte` do código.

**c)** Repita o experimento para

$$
N=2,\;4,\;8,\;16,\;32.
$$

**d)** Compare o erro máximo entre duas malhas consecutivas.

O que acontece com o erro quando $h$ é dividido por $2$?

A partir dos resultados numéricos, investigue a ordem de convergência do método.

In [17]:
#!jupyter nbconvert --to html --embed-images Aula_2_Fluidos.ipynb